# Administrative & Governance Dataset — Kalomo Town Council

**Owner:** Nicholas &nbsp;|&nbsp; **CSC4792 Mini Project, Group project — Kalomo Town Council, Zambia**

This notebook documents how `db-unza26-csc4792-kalomo_town_council_administrative_governance.csv` was built: council leadership, departments, ward-level representation, grassroots governance committees, constituencies and chiefdoms.

The two supporting scripts referenced throughout are:
- `scripts/scraping/scrape_administrative_governance.py`
- `scripts/cleaning/clean_administrative_governance.py`

This notebook works offline once the repo is cloned — it inspects the files those scripts produced rather than re-scraping the live site on every run (see Step 2 for why).

## Step 1 — Manually browsing the site first

Before writing any scraping code, the council site was browsed by hand to find where administrative/governance content lives. Unlike the development-plans dataset (a handful of downloadable PDFs), this data is spread across several ordinary HTML pages plus individual news posts:

- **About Us** (`?page_id=118`) — district administrative history
- **Read more / district profile** (`?page_id=2242`) — wards, chiefdoms, constituencies
- **Civic Leaders** (`?page_id=2871`) — ward councillors, grouped by constituency
- **Departments** (`?page_id=770`) — council department listing
- **FAQs** (`?page_id=2259`) — defines CDF and WDC
- Several individual **news posts** (`?p=...`) that name office holders (Council Chairperson, Council Secretary, Directors) and describe grassroots governance structures (WDCs, CWACs, SDMCs, Headmen) in action

This manual pass produced the `SOURCE_PAGES` dict in the scraping script below.

## Step 2 — Automated scraping, and why this session used indexed snapshots instead

`scripts/scraping/scrape_administrative_governance.py` is a normal `requests` + `BeautifulSoup` scraper (same pattern as the other three datasets in this project — including the `verify=False` workaround for the council site's known incomplete TLS chain, see `scrape_development_plans.py`). It fetches each page in `SOURCE_PAGES`, saves the raw HTML, and extracts visible text.

**What actually happened in this session:** the sandboxed environment used to build this repo could not reach `kalomocouncil.gov.zm` directly (`requests.get` returned HTTP 403; hosted fetch tooling in this environment separately honours the site's `robots.txt`, which disallows automated access). Re-running the script from an unrestricted connection — a teammate's laptop, as used for the other datasets — will populate `data/raw/administrative_governance/` with full HTML and text directly, which the script already supports.

For this session, the equivalent content was instead compiled by hand into `data/raw/administrative_governance/source_pages_extract.txt`, page by page, from indexed copies of the exact same pages and from the council's own news posts (which were fully readable via search indexing even though live fetches were not). This follows the same principle used throughout this project for the scanned/rotated PDFs in the development-plans dataset: record what a source actually contains, and say plainly where it doesn't.

In [1]:
import sys, os
sys.path.append(os.path.join("..", "scripts", "scraping"))
import scrape_administrative_governance as sag

print("Source pages targeted:")
for name, url in sag.SOURCE_PAGES.items():
    print(f" - {name}: {url}")

Source pages targeted:
 - about_us: https://www.kalomocouncil.gov.zm/?page_id=118
 - district_profile: https://www.kalomocouncil.gov.zm/?page_id=2242
 - civic_leaders: https://www.kalomocouncil.gov.zm/?page_id=2871
 - departments: https://www.kalomocouncil.gov.zm/?page_id=770
 - faqs: https://www.kalomocouncil.gov.zm/?page_id=2259
 - news_index: https://www.kalomocouncil.gov.zm/?page_id=187
 - news_cdf_equipment_commissioning: https://www.kalomocouncil.gov.zm/?p=1799
 - news_2026_budget_consultative_meeting: https://www.kalomocouncil.gov.zm/?p=3782
 - news_mis_launch: https://www.kalomocouncil.gov.zm/?p=4672
 - news_cash_for_work_sensitization: https://www.kalomocouncil.gov.zm/?p=5104


## Step 3 — Problems encountered

**Access restrictions beyond the known TLS issue.** As on the other datasets, the council server needs `verify=False`. Additionally, in this session specifically, the sandboxed tooling used to build the repo could not reach the site at all (see Step 2) — this is an environment restriction, not evidence the site itself blocks scraping (the other three datasets in this project were scraped successfully from a normal connection).

**Incomplete page content via indexed snapshots.** The *Departments* page (`?page_id=770`) and the *Civic Leaders* page (`?page_id=2871`) both use client-side rendering for parts of their content (tabs/accordions, similar to the Publications page used in the development-plans dataset). Indexed snapshots captured only site chrome for Departments, and only one of the ~20 ward councillor entries for Civic Leaders. The Civic Leaders gap was closed with a direct manual browser capture of the page (2026-09-11), which yielded the full 20-ward roster split across the district's two constituencies (8 in Dundumwezi, 12 in Kalomo Central) plus the Council Chairperson and Vice Council Chairperson — the Departments gap remains open and is recorded honestly in `DATA_DICTIONARY.md` and `source_pages_extract.txt` rather than papered over.

**A Council Secretary discrepancy, resolved by dating the evidence rather than picking one name.** Two different individuals are documented as Kalomo's Council Secretary in different sources: Lisa Mpasela (2023 IDP launch, and a CDF-project news post referencing the 2023 funding phase) and Trophius Kufanga (the council's Web-Based MIS launch article — undated, but referencing 2026-era infrastructure). Cross-referencing found that a Trophius Kufanga was separately Council Secretary at **Masaiti** Town Council per a dated 2023 public notice — consistent with the routine practice of council secretaries transferring between local authorities. Rather than guessing which name is "current", both are recorded as rows with the supporting context and the caveat that the exact handover date isn't stated in either source.

In [2]:
raw_dir = os.path.join("..", "data", "raw", "administrative_governance")
notes_path = os.path.join(raw_dir, "source_pages_extract.txt")
with open(notes_path, encoding="utf-8") as f:
    notes = f.read()
print(f"{notes_path}: {len(notes)} chars of compiled source notes")
print()
print(notes[:1200], "...")

../data/raw/administrative_governance/source_pages_extract.txt: 16225 chars of compiled source notes

SOURCE PAGE EXTRACTS - Kalomo Town Council administrative/governance dataset
Collected by: Nicholas
Method: scripts/scraping/scrape_administrative_governance.py targets these
exact pages on https://www.kalomocouncil.gov.zm/. The council site blocks
automated/agent HTTP clients (returns 403 to a plain requests.get from this
environment, and is robots-disallowed for hosted fetch tools), so the
running notes below were compiled by cross-referencing indexed copies of
each page's rendered text (the same pages the script targets) together with
the council's own news posts, which are separately indexed and were fully
readable. This mirrors the approach already used for the scanned/rotated
PDFs in data/raw/development_plans (documented from listing/context rather
than full text) - content is recorded honestly per source, with gaps noted
rather than invented. Running the script from a normal re

## Step 4 — Structuring the notes into rows

As with the development-plans dataset, this content is prose, not tables, so `source_pages_extract.txt` was read by hand and each governance fact pulled out into `clean_administrative_governance.py`. Six row categories were used:

1. **Leadership** — Council Chairperson, Council Secretary (both documented, see Step 3), District Commissioner (flagged as a central-government, not Council, office), Directors of Finance/Engineering/ICT, and the district's two Members of Parliament (flagged as national, not Council, offices — included because the council's own site organises its Civic Leaders page around these two constituencies).
2. **Department** — the four departments/offices directly confirmed by name in council sources (Council Secretary's office, Finance, Engineering, ICT). Departments typical of comparable Zambian councils (Public Health, Planning, Human Resource) were deliberately **not** added without direct confirmation.
3. **Ward** — the one ward councillor confirmed via indexed content.
4. **Committee** — the grassroots governance bodies named together in the council's own R-CFW sensitization news post: Ward Development Committees, Community Welfare Assistance Committees, Satellite Disaster Management Committees, and Headmen, each present across all 20 wards.
5. **Constituency** — Kalomo Central and Dundumwezi.
6. **Chiefdom** — Chikanta, Siachitema, Sipatunyana.

`record_type` extends the illustrative list in `docs/DATA_DICTIONARY.md` (which was always marked "e.g.") to match what the sources actually supported, and the dictionary was updated to document the change and the coverage caveat.

**Standardisation applied**, matching the conventions agreed for every dataset in this project:
- `record_id` assigned sequentially as `AG-001` … `AG-023`
- Every text field stripped of whitespace; empty/`nan`/`None` normalised to `"N/A"`
- Duplicate rows dropped on `(name_or_title, source_url)`
- Every row asserted to have a working `https://` `source_url` and a unique `record_id`
- `date_scraped` stamped with the date the cleaning script was run
- Output written with `sep="|"` per the project's naming/format convention

## Step 5 — Loading and inspecting the final dataset

In [3]:
import pandas as pd

csv_path = os.path.join("..", "data", "processed", "db-unza26-csc4792-kalomo_town_council_administrative_governance.csv")
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
df.head(10)

,record_id,record_type,name_or_title,role_or_function,ward,date,source_url,date_scraped
0,AG-001,Contact,Kalomo Town Council - General Contact Information,Email: towncouncilkalomo@gmail.com | Address: ...,N/A,N/A,https://www.kalomocouncil.gov.zm/?page_id=770,2026-09-12
1,AG-002,Leadership,Coy Makaya,Council Chairperson,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
2,AG-003,Leadership,Lisa Mpasela,Council Secretary (per the 2023 IDP launch and...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=1799,2026-09-12
3,AG-004,Leadership,Trophius Kufanga,Council Secretary (per the council's Web-Based...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
4,AG-005,Leadership,Joshua Munsaka Sikaduli,District Commissioner (Office of the President...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
5,AG-006,Leadership,Jimmy Mubanga,Director of Finance,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
6,AG-007,Leadership,Joel Mweempe,Director of Engineering,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=3782,2026-09-12
7,AG-008,Leadership,Judith Beene,Director of Information and Communication Tech...,N/A,N/A,https://www.kalomocouncil.gov.zm/?p=4672,2026-09-12
8,AG-009,Leadership,Harry Kamboni,"Member of Parliament, Kalomo Central Constitue...",N/A,N/A,https://en.wikipedia.org/wiki/Kalomo_Central,2026-09-12
9,AG-010,Leadership,Valencia Simwale,Vice Council Chairperson; also serves as the N...,Naluja Ward,N/A,https://www.kalomocouncil.gov.zm/?page_id=2871,2026-09-12


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   record_id         50 non-null     str  
 1   record_type       50 non-null     str  
 2   name_or_title     50 non-null     str  
 3   role_or_function  50 non-null     str  
 4   ward              50 non-null     str  
 5   date              50 non-null     str  
 6   source_url        50 non-null     str  
 7   date_scraped      50 non-null     str  
dtypes: str(8)
memory usage: 3.3 KB


In [5]:
print("Rows by record_type:")
print(df["record_type"].value_counts())
print("\nRows with ward = N/A:", (df["ward"] == "N/A").sum(), "out of", len(df))
print("Rows with date = N/A:", (df["date"] == "N/A").sum(), "out of", len(df))

Rows by record_type:
record_type
Ward            20
Leadership      10
Department       9
Committee        4
Chiefdom         3
Constituency     2
Contact          1
Service          1
Name: count, dtype: int64

Rows with ward = N/A: 25 out of 50
Rows with date = N/A: 46 out of 50


## Limitations

- Ward councillor roster is now complete (20/20 wards, manually captured directly from the live Civic Leaders page on 2026-09-11 after indexed retrieval fell short — see Step 3).
- No confirmed department directory beyond the 4 departments directly named in council sources; other departments typical of comparable councils were deliberately excluded rather than assumed.
- Most `date` values remain `"N/A"` — the council's news posts and pages mostly omit explicit publish dates in their indexed text. A few dates were recovered where the source's own dated feed was visible (e.g. the Institutional Management page's activity feed confirmed 16 June 2026 for the R-CFW sensitization post, and a separate news post confirmed 11 September 2024 for the Food Security Pack Program).
- The Council Secretary discrepancy (Lisa Mpasela vs. Trophius Kufanga) is presented as two dated-as-best-as-possible rows with explicit caveats rather than resolved to a single "current" answer, since neither source states an exact handover date.
- Members of Parliament are included for completeness of the district's civic-leadership picture (the council's own Civic Leaders page is organised by constituency) but are explicitly flagged as national, not Council, offices.

See `docs/DATA_DICTIONARY.md` for the full column reference for this and every other dataset in this project, including the coverage note added for this dataset.